> 📎 **Appendix notebook — reference style.** This is one of the optional appendices (see `README.md`). Unlike the main course notebooks, appendices are written as a demo / reference: they focus on *seeing* a library at work rather than on interactive exercises. You won't find the full Solution / Debug-me / Self-assessment scaffolding here — and if you don't have the optional dependency installed, the notebook will stop at the first import. That's by design.

---
# 📓 Notebook A3 — RAG & Agent Frameworks

> **Module:** AI Engineering · **Type:** Appendix · **Estimated time:** 45–60 min · **Difficulty:** Intermediate

Notebook 18 implemented RAG by hand. Notebook 19 implemented tool-using agents by hand. Both were ~150 lines of clearly-readable Python. So why do **LangChain**, **LlamaIndex**, **Haystack**, **DSPy**, **AutoGen**, **CrewAI** and friends exist — and when should *you* reach for them?

The answer in one line: **when your pipeline grows past hand-rolled and you want batteries — loaders, retrievers, tracing, evaluation, deploy — without writing them yourself.**

This notebook gives you a code-level tour of the major frameworks against the **same RAG task** so you can compare like-for-like.

---

## 🎯 Learning objectives
- Map every framework to a job-to-be-done: orchestration, retrieval depth, programmatic prompts, multi-agent coordination.
- Read framework code without panicking — see the same RAG pipeline in 5 different libraries.
- Pick a framework using a 3-question rubric (or *not* pick one and ship faster).

## ✅ Prerequisites
- Notebook 18 (RAG by hand) and Notebook 19 (tools & agents).
- Notebook A2 (vector stores) helps but is not required.


## 1. The shared task

> *Given a tiny FAQ corpus, answer a user's question with citations.*

All five framework snippets below do the **same thing**: embed → retrieve top-3 → stuff into a prompt → call the LLM → return text with sources. We'll do it once in plain Python first, as a baseline.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

# Make the repo-root llm_providers.py importable regardless of which folder
# Jupyter launched the kernel in (it lives at the course root).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / "llm_providers.py").exists():
        sys.path.insert(0, str(_p)); break

from llm_providers import MockLLM, MockEmbedder
import numpy as np

DOCS = [
    ("billing-1",  "We accept Visa, Mastercard, Amex. Update your card under Settings → Billing."),
    ("billing-2",  "Subscriptions renew monthly on the same day you signed up."),
    ("billing-3",  "Refunds are pro-rated for unused days within 30 days of the charge."),
    ("account-1",  "To reset your password, click 'Forgot password' on the sign-in screen."),
    ("account-2",  "Email changes require confirmation from both the old and the new address."),
    ("product-1",  "Export your data as CSV under Settings → Export. Limit: 10 MB per file."),
    ("product-2",  "Usage statistics live under Dashboard → Usage and refresh every hour."),
    ("support-1",  "Live chat is available Mon-Fri, 09:00-18:00 UTC."),
]
ids   = [d[0] for d in DOCS]
texts = [d[1] for d in DOCS]

embedder = MockEmbedder(dim=128)
llm      = MockLLM()

doc_v = embedder.embed(texts)
print("indexed", len(texts), "docs · dim =", embedder.dim)


indexed 8 docs · dim = 128


## 2. Baseline — RAG in plain Python (the NB18 recipe)

This is the *yardstick*. Every framework below replaces parts of this with a class hierarchy.


In [2]:
def retrieve(query: str, k: int = 3):
    q = embedder.embed([query])[0]
    sims = doc_v @ q
    order = np.argsort(-sims)[:k]
    return [(ids[i], texts[i], float(sims[i])) for i in order]

def answer(query: str, k: int = 3):
    hits = retrieve(query, k)
    context = "\n".join(f"[{i+1}] ({h[0]}) {h[1]}" for i, h in enumerate(hits))
    prompt = (f"Answer the user's question using ONLY the context. "
              f"Cite sources like [1], [2].\n\nContext:\n{context}\n\nQuestion: {query}")
    resp = llm.chat(messages=[
        {"role":"system","content":"You answer with citations."},
        {"role":"user",  "content":prompt},
    ])
    return resp["text"], [h[0] for h in hits]

text, sources = answer("Can I get my money back?")
print("Answer :", text)
print("Sources:", sources)


Answer : Based on the provided context, here is a short answer. You asked: Answer the user's question using ONLY the context.
Sources: ['account-1', 'product-1', 'billing-2']


## 3. LangChain — the orchestration layer

[**LangChain**](https://python.langchain.com/) is the most popular framework. Think of it as **plumbing**: it gives you typed wrappers around prompts, retrievers, LLMs, output parsers, memory, and tracing — and a way to compose them with the `|` pipe operator.

```bash
pip install langchain langchain-community langchain-openai chromadb
```

The same RAG pipeline in LangChain Expression Language (LCEL):


In [3]:
# ── LangChain reference code (uncomment after installing the packages) ───
# from langchain_community.vectorstores import Chroma
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.runnables import RunnablePassthrough
#
# vs = Chroma.from_texts(texts, embedding=OpenAIEmbeddings(),
#                        metadatas=[{"id": i} for i in ids])
# retriever = vs.as_retriever(search_kwargs={"k": 3})
#
# template = "Answer using ONLY the context. Cite [1], [2]...\n\nContext: {context}\nQuestion: {question}"
#
# prompt = ChatPromptTemplate.from_template(template)
# chain = (
#     {"context": retriever, "question": RunnablePassthrough()}
#     | prompt
#     | ChatOpenAI(model="gpt-4o-mini", temperature=0)
#     | StrOutputParser()
# )
# print(chain.invoke("Can I get my money back?"))

# Mental model offline-equivalent:
def langchain_like_chain(question):
    docs = retrieve(question)
    formatted = "\n".join(f"[{i+1}] {d[1]}" for i, d in enumerate(docs))
    prompt_text = f"Context:\n{formatted}\nQ: {question}"
    return llm.chat(messages=[{"role":"user","content":prompt_text}])["text"]

print(langchain_like_chain("Can I get my money back?"))


This is a mock response. Plug a real LLM in to get a real answer.


**What LangChain buys you**
- A *huge* ecosystem of integrations (every vector store, every LLM, every loader).
- LCEL composability with streaming, async, batch, retries out of the box.
- `LangSmith` for tracing and evaluation.

**What it costs**
- Heavy abstraction; the docs change faster than the API.
- Stack traces inside chains can be daunting.
- For very simple tasks, the framework lines outnumber the business logic.


## 4. LlamaIndex — retrieval-first

[**LlamaIndex**](https://docs.llamaindex.ai/) (formerly GPT-Index) specialises in **ingestion and retrieval**. It has the strongest off-the-shelf set of retrievers: hybrid (BM25 + vector), fusion (RRF), tree, knowledge-graph, hierarchical, auto-merging.

```bash
pip install llama-index llama-index-llms-openai
```


In [4]:
# ── LlamaIndex reference code ────────────────────────────────────────────
# from llama_index.core import VectorStoreIndex, Document, Settings
# from llama_index.llms.openai import OpenAI
#
# Settings.llm = OpenAI(model="gpt-4o-mini")
# docs = [Document(text=t, metadata={"id": i}) for i, t in zip(ids, texts)]
# index = VectorStoreIndex.from_documents(docs)
# query_engine = index.as_query_engine(similarity_top_k=3)
# response = query_engine.query("Can I get my money back?")
# print(response.response)
# print("Sources:", [n.metadata["id"] for n in response.source_nodes])

# Mental model: LlamaIndex's RetrieverQueryEngine ≈ retrieve + synthesize.
# The synthesize step has multiple strategies: refine, tree_summarize, compact.
print("LlamaIndex shines on advanced retrieval — see below.")


LlamaIndex shines on advanced retrieval — see below.


**Advanced retrieval moves you actually use**

| Pattern | When | LlamaIndex name |
|---|---|---|
| Hybrid (BM25 + vector) | Exact-match keywords matter (codes, IDs) | `QueryFusionRetriever` |
| Reciprocal rank fusion | Multiple retrievers disagree | `QueryFusionRetriever(mode="reciprocal_rerank")` |
| Auto-merging | Big docs; merge sibling chunks | `AutoMergingRetriever` |
| Sub-question | Question contains 2+ independent asks | `SubQuestionQueryEngine` |
| Recursive / hierarchical | Wikis with section trees | `RecursiveRetriever` |
| Re-ranking with a cross-encoder | Final-pass quality boost | `LLMRerank` or `SentenceTransformerRerank` |


## 5. Haystack — production pipelines

[**Haystack 2**](https://docs.haystack.deepset.ai/) (by deepset) is the most *production-flavoured*: strongly-typed components, YAML pipelines, REST deployment, evaluation tooling.

```bash
pip install haystack-ai
```


In [5]:
# ── Haystack 2 reference code ────────────────────────────────────────────
# from haystack import Pipeline, Document
# from haystack.components.embedders import OpenAITextEmbedder, OpenAIDocumentEmbedder
# from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
# from haystack.components.builders import PromptBuilder
# from haystack.components.generators import OpenAIGenerator
# from haystack.document_stores.in_memory import InMemoryDocumentStore
#
# store = InMemoryDocumentStore()
# doc_embedder = OpenAIDocumentEmbedder()
# store.write_documents(doc_embedder.run([Document(content=t, meta={"id": i})
#                                         for i, t in zip(ids, texts)])["documents"])
#
# rag = Pipeline()
# rag.add_component("q_embedder", OpenAITextEmbedder())
# rag.add_component("retriever",  InMemoryEmbeddingRetriever(store, top_k=3))
# rag.add_component("prompt",     PromptBuilder(template="Context:\n{{ docs }}\nQ: {{ q }}"))
# rag.add_component("llm",        OpenAIGenerator(model="gpt-4o-mini"))
# rag.connect("q_embedder.embedding", "retriever.query_embedding")
# rag.connect("retriever.documents",  "prompt.docs")
# rag.connect("prompt.prompt",        "llm.prompt")
# print(rag.run({"q_embedder": {"text": "Can I get my money back?"},
#                "prompt":     {"q":    "Can I get my money back?"}}))
print("Haystack pipelines compose as DAGs with typed connectors.")


Haystack pipelines compose as DAGs with typed connectors.


## 6. DSPy — programmatic prompting

[**DSPy**](https://dspy-docs.vercel.app/) (Stanford) is the odd one out. Instead of writing prompts by hand it lets you **declare the I/O signature** of a step, then optimises the prompt against a metric using examples.

```python
# pip install dspy-ai
# import dspy
# dspy.settings.configure(lm=dspy.OpenAI(model="gpt-4o-mini"))
#
# class FAQAnswer(dspy.Signature):
#     "Answer using the retrieved context. Cite sources."
#     context: str = dspy.InputField()
#     question: str = dspy.InputField()
#     answer: str = dspy.OutputField()
#
# rag = dspy.ChainOfThought(FAQAnswer)
# pred = rag(context="\n".join(texts[:3]), question="Can I get my money back?")
# print(pred.answer)
```

DSPy is what you reach for when you want to **train your prompts** the way you train models: a few labelled examples + a metric + an optimiser (`BootstrapFewShot`, `MIPRO`). Not a competitor to LangChain — they compose.


## 7. Agent frameworks — when one LLM call isn't enough

When the task involves **tool use and multi-step planning**, you typically pick from these:

| Framework | Mental model | Strength |
|---|---|---|
| **LangGraph** (LangChain) | A state machine over LCEL nodes | Persistence, human-in-the-loop, replay |
| **AutoGen** (Microsoft) | Multiple LLM agents talking to each other | Easy multi-agent debates, group chats |
| **CrewAI** | Role-playing crews (researcher, writer, critic) | Clean ergonomics for prototypes |
| **smolagents** (HuggingFace) | Code-writing agents in pure Python | Tiny dependency footprint |
| **OpenAI Agents SDK** | Function-calling driven runs with handoffs | First-party, simplest if you're already on OpenAI |

```python
# smolagents — agents that write & execute Python:
# pip install smolagents
# from smolagents import CodeAgent, ToolCallingAgent, tool, HfApiModel
# @tool
# def lookup_faq(query: str) -> str:
#     "Return the closest FAQ entry to the query."
#     return retrieve(query)[0][1]
# agent = CodeAgent(tools=[lookup_faq], model=HfApiModel())
# print(agent.run("Tell me about refunds and password resets."))
```

```python
# AutoGen — multi-agent group chat:
# from autogen import AssistantAgent, UserProxyAgent
# planner = AssistantAgent("planner", system_message="Plan the answer.", llm_config={...})
# writer  = AssistantAgent("writer",  system_message="Write the answer.", llm_config={...})
# user    = UserProxyAgent("user", code_execution_config=False)
# user.initiate_chat(planner, message="Can I get my money back?")
```

Course philosophy reminder: **Notebook 19** showed you how a tool-using agent works *underneath* these abstractions. Reach for an agent framework when you need multi-agent coordination, durable state, or human-in-the-loop — not for "call a couple of tools".


## 8. Decision rubric — three questions

```
Q1. Is the pipeline > ~200 lines of glue code?
       NO  → stay hand-rolled.  Frameworks add weight you don't need.
       YES → continue

Q2. Is retrieval the hard part (advanced retrievers, ingestion of many formats)?
       YES → LlamaIndex
       NO  → continue

Q3. Will I deploy this as a service with strong typing & eval?
       YES → Haystack 2
       NO  → LangChain  (or LangGraph if it's an agent)

Side question — do I want to *optimise* my prompts with data?
       YES → DSPy  (composes on top of either of the above)
```

Practical advice: when in doubt, **start hand-rolled**. The frameworks integrate well later, but ripping one *out* is harder than putting one *in*.


## 🧪 Exercises

### Exercise 1 — Translate the baseline to LangChain-style LCEL
Take the `answer()` function from §2 and refactor it into a **dict-pipe-prompt-pipe-llm-pipe-parser** chain using just `dict`, `lambda`, and `functools.reduce`. The point is to *feel* the LCEL mental model without installing LangChain.


In [6]:
# ── Exercise 1 solution ──────────────────────────────────────────────────
from functools import reduce

def make_step(fn): return fn
def compose(*steps): return reduce(lambda f, g: (lambda x: g(f(x))), steps)

retrieve_step = make_step(lambda q: {"q": q, "ctx": retrieve(q)})
format_step   = make_step(lambda d: {"q": d["q"],
                                      "prompt": "Context:\n" +
                                                "\n".join(f"[{i+1}] {h[1]}" for i, h in enumerate(d["ctx"]))
                                                + f"\nQ: {d['q']}",
                                      "sources": [h[0] for h in d["ctx"]]})
llm_step      = make_step(lambda d: {"text": llm.chat(messages=[{"role":"user","content":d["prompt"]}])["text"],
                                      "sources": d["sources"]})
parser_step   = make_step(lambda d: d["text"] + " — sources: " + ", ".join(d["sources"]))

chain = compose(retrieve_step, format_step, llm_step, parser_step)
print(chain("Can I get my money back?"))


This is a mock response. Plug a real LLM in to get a real answer. — sources: account-1, product-1, billing-2


### Exercise 2 — Frame an agent task as a state machine
Without writing any framework code, design (on paper) a 4-state machine for an agent that: (1) classifies a customer email, (2) retrieves relevant policy, (3) drafts a reply, (4) asks a human to approve. Identify the *transition condition* out of each state. This is exactly what LangGraph nodes encode.


In [7]:
# ── Exercise 2 solution (sketch with stub functions) ─────────────────────
states = {"CLASSIFY", "RETRIEVE", "DRAFT", "APPROVE", "DONE"}

def transitions(state, payload):
    if state == "CLASSIFY":
        return ("RETRIEVE", payload) if payload.get("category") else ("DONE", {"error": "unknown"})
    if state == "RETRIEVE":
        return ("DRAFT", payload) if payload.get("hits") else ("DONE", {"error": "no_kb_match"})
    if state == "DRAFT":
        return ("APPROVE", payload) if payload.get("draft") else ("DRAFT", payload)
    if state == "APPROVE":
        return ("DONE", payload) if payload.get("approved") else ("DRAFT", {**payload, "draft": None})
    return ("DONE", payload)

# Toy walk-through
state, payload = "CLASSIFY", {"category": "refund"}
trace = [state]
while state != "DONE":
    payload.setdefault("hits", True); payload.setdefault("draft", "Sorry to hear...")
    payload.setdefault("approved", True)
    state, payload = transitions(state, payload)
    trace.append(state)
print("Trace:", " → ".join(trace))


Trace: CLASSIFY → RETRIEVE → DRAFT → APPROVE → DONE


## 🧠 Key takeaways

- Frameworks are **glue + integrations + ops**, not magic. The math is identical to NB18/NB19.
- **LangChain** for breadth, **LlamaIndex** for retrieval depth, **Haystack** for production typing, **DSPy** for prompt optimisation, **LangGraph/AutoGen/CrewAI/smolagents** for agent orchestration.
- Stay hand-rolled until the pipeline complexity *demands* a framework — adding one is easier than removing one.
- The hardest framework decision is **vector store + retriever**; everything else is a thin top layer.

## 🚀 Next step

If you haven't already, work through **A2** (vector-store landscape). Then return to your real project and pick at most **one** framework — only if you've already hit the pain it solves.
